In [1]:
# =============================================================================
# IMPORTACAO DE DADOS GPKG -> PostgreSQL  |  AULA 12
# Sobreposicao com categorias fundiarias restritivas
# =============================================================================
import geopandas as gpd
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# =============================================================================
# CONFIGURACOES - ALTERE CONFORME NECESSARIO
# =============================================================================

# Pasta onde estao os arquivos GPKG
PASTA_DADOS = Path(r"C:\Temp")

# =============================================================================
# ARQUIVOS GPKG
# =============================================================================
CAR_MT          = "es_mt_car_20260406.gpkg"
UC              = "pa_br_ucs_mma_2026.gpkg"
TI              = "pa_br_terrasindigenas_funai_2026.gpkg"
TQ              = "pa_br_territoriosquilombolas_incra_2026.gpkg"
CNFP            = "pa_br_cnfp_sfb_2024_retificado.gpkg"
ASSENTAMENTOS   = "pa_br_assentamentos_incra_2026.gpkg"
MALHA_MT        = "pa_mt_malhafundiaria_2025_cdt.gpkg"
MODULOS_FISCAIS = "pa_br_modulosfiscais_incra.gpkg"
SIGEF           = "pa_br_sigef_privado_incra_2026.gpkg"
SNCI            = "pa_br_snci_privado_incra_2026.gpkg"

# =============================================================================
# CONTROLE DE IMPORTACAO
# True  = importa (cria/substitui a tabela)
# False = pula
# =============================================================================
IMPORTAR_CAR_MT          = True
IMPORTAR_UC              = True
IMPORTAR_TI              = True
IMPORTAR_TQ              = True
IMPORTAR_CNFP            = True
IMPORTAR_ASSENTAMENTOS   = True
IMPORTAR_MALHA_MT        = True
IMPORTAR_MODULOS_FISCAIS = True
IMPORTAR_SIGEF           = True
IMPORTAR_SNCI            = True

# Se False, mantem a tabela existente no banco (nao reimporta)
RECRIAR_TABELAS = False

# =============================================================================
# CREDENCIAIS DO BANCO LOCAL (geotec)
# =============================================================================
DB_HOST     = "localhost"
DB_PORT     = "5432"
DB_USER     = "postgres"
DB_PASSWORD = "postgres"
DB_NAME     = "geotec"

# =============================================================================
# MAPEAMENTO: arquivo GPKG -> (schema, tabela)
# =============================================================================
IMPORTACOES = [
    (CAR_MT,          "car",    "es_mt_car_20260406",                     IMPORTAR_CAR_MT),
    (UC,              "mma",    "pa_br_ucs_mma_2026",                     IMPORTAR_UC),
    (TI,              "funai",  "pa_br_terrasindigenas_funai_2026",        IMPORTAR_TI),
    (TQ,              "incra",  "pa_br_territoriosquilombolas_incra_2026", IMPORTAR_TQ),
    (CNFP,            "sfb",    "pa_br_cnfp_sfb_2024_retificado",          IMPORTAR_CNFP),
    (ASSENTAMENTOS,   "incra",  "pa_br_assentamentos_incra_2026",         IMPORTAR_ASSENTAMENTOS),
    (MALHA_MT,        "malha",  "pa_mt_malhafundiaria_2025_cdt",          IMPORTAR_MALHA_MT),
    (MODULOS_FISCAIS, "incra",  "pa_br_modulosfiscais_incra",             IMPORTAR_MODULOS_FISCAIS),
    (SIGEF,           "incra",  "pa_br_sigef_privado_incra_2026",         IMPORTAR_SIGEF),
    (SNCI,            "incra",  "pa_br_snci_privado_incra_2026",          IMPORTAR_SNCI),
]

# =============================================================================
# FUNCOES
# =============================================================================

def conectar_banco():
    return create_engine(
        f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    )


def criar_schemas(engine):
    schemas = ["car", "mma", "funai", "incra", "sfb", "malha"]
    with engine.connect() as conn:
        for s in schemas:
            conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {s}"))
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))
        conn.commit()
    print(f"[OK] Schemas prontos: {', '.join(schemas)}")


def tabela_existe(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            conn.execute(text(f"SELECT 1 FROM {schema}.{tabela} LIMIT 1"))
            return True
    except Exception:
        return False


def contar_registros(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            r = conn.execute(text(f"SELECT COUNT(*) FROM {schema}.{tabela}"))
            return r.fetchone()[0]
    except Exception:
        return 0


def importar_gpkg(engine, caminho, schema, tabela):
    print(f"  Lendo: {caminho.name}")
    gdf = gpd.read_file(str(caminho))
    print(f"  Registros: {len(gdf)}")

    tem_geometria = (
        isinstance(gdf, gpd.GeoDataFrame)
        and gdf.active_geometry_name is not None
        and not gdf.geometry.isna().all()
    )

    if tem_geometria:
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4674)
        else:
            gdf = gdf.to_crs(epsg=4674)
        gdf.to_postgis(
            name=tabela, con=engine, schema=schema,
            if_exists="replace", index=False
        )
    else:
        df = pd.DataFrame(gdf)
        geom_col = getattr(gdf, "active_geometry_name", None)
        if geom_col:
            df = df.drop(columns=geom_col, errors="ignore")
        print("  [INFO] Arquivo sem geometria ativa; importando como tabela comum")
        df.to_sql(
            name=tabela, con=engine, schema=schema,
            if_exists="replace", index=False
        )

    print(f"  OK -> {schema}.{tabela}")


def importar_se_necessario(engine, arquivo, schema, tabela, ativo):
    if not ativo:
        print(f"  [SKIP] {schema}.{tabela} (desativado)")
        return

    caminho = PASTA_DADOS / arquivo
    if not caminho.exists():
        print(f"  [ERRO] Arquivo nao encontrado: {caminho}")
        return

    if tabela_existe(engine, schema, tabela) and not RECRIAR_TABELAS:
        n = contar_registros(engine, schema, tabela)
        print(f"  [EXISTE] {schema}.{tabela} ({n} registros) - mantido")
        return

    importar_gpkg(engine, caminho, schema, tabela)


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("=" * 65)
    print("IMPORTACAO DE DADOS - AULA 12")
    print("=" * 65)

    if not PASTA_DADOS.exists():
        print(f"[ERRO] Pasta nao encontrada: {PASTA_DADOS}")
        return

    engine = conectar_banco()
    print("[OK] Conectado ao PostgreSQL")

    criar_schemas(engine)

    for i, (arquivo, schema, tabela, ativo) in enumerate(IMPORTACOES, 1):
        print(f"\n[{i}/{len(IMPORTACOES)}] {schema}.{tabela}")
        importar_se_necessario(engine, arquivo, schema, tabela, ativo)

    print("\n" + "=" * 65)
    print("RESUMO DAS TABELAS")
    print("=" * 65)
    for arquivo, schema, tabela, ativo in IMPORTACOES:
        if ativo:
            n = contar_registros(engine, schema, tabela)
            print(f"  {schema}.{tabela}: {n} registros")

    engine.dispose()
    print("\n[OK] Importacao concluida!")
    print("=" * 65)


main()

IMPORTACAO DE DADOS - AULA 12
[OK] Conectado ao PostgreSQL
[OK] Schemas prontos: car, mma, funai, incra, sfb, malha

[1/10] car.es_mt_car_20260406
  [EXISTE] car.es_mt_car_20260406 (213912 registros) - mantido

[2/10] mma.pa_br_ucs_mma_2026
  [EXISTE] mma.pa_br_ucs_mma_2026 (157 registros) - mantido

[3/10] funai.pa_br_terrasindigenas_funai_2026
  [EXISTE] funai.pa_br_terrasindigenas_funai_2026 (94 registros) - mantido

[4/10] incra.pa_br_territoriosquilombolas_incra_2026
  [EXISTE] incra.pa_br_territoriosquilombolas_incra_2026 (7 registros) - mantido

[5/10] sfb.pa_br_cnfp_sfb_2024_retificado
  [EXISTE] sfb.pa_br_cnfp_sfb_2024_retificado (2010 registros) - mantido

[6/10] incra.pa_br_assentamentos_incra_2026
  [EXISTE] incra.pa_br_assentamentos_incra_2026 (632 registros) - mantido

[7/10] malha.pa_mt_malhafundiaria_2025_cdt
  [EXISTE] malha.pa_mt_malhafundiaria_2025_cdt (8237 registros) - mantido

[8/10] incra.pa_br_modulosfiscais_incra
  [EXISTE] incra.pa_br_modulosfiscais_incra (557